In [1]:
"""
将CB Dynamic的模型特征重要程度进行排序,画成heatmap
"""
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from catboost import CatBoostRegressor, Pool

In [2]:
# =========================================================
# 基本参数
# =========================================================
IN_PATH = "/data2/zhoujb/project/cropVarCefComb/processData/9-0-runOnline/trainTestData/"

OUTDIR = os.path.join(IN_PATH, "shap_rank_heatmap_sidecolor")
os.makedirs(OUTDIR, exist_ok=True)

traits = [
    "Yield_per_plant",
    "1000-grain_weight",
    "Grain_length_to_width_ratio",
    "Heading_date",
    "Plant_height",
    "Grain_per_panicle",
    "Seed_set_rate",
    "Tiller_number"
]

tmp_map = pd.read_table("/data2/zhoujb/project/cropVarAppr/processData/gene_coexpression_info.txt")
tmp_map = tmp_map[["MSU_id", "Type"]]
tmp_map["MSU_id"] = tmp_map["MSU_id"].str.replace(r"\.\d+$", "", regex=True)
tmp_map_d = tmp_map.set_index("MSU_id").to_dict()["Type"]

train_file = os.path.join(IN_PATH, f"train-{traits[0]}.txt")
train_df = pd.read_table(train_file)

tmp_feat_list = [x[4:] for x in train_df.columns if x.startswith("fea_")] + [traits[0]]
tmp_feat_tuple = []
for item in tmp_feat_list:
    if item.startswith("LOC_"):
        tmp_feat_tuple.append([item, tmp_map_d[item]])
    else:
        tmp_feat_tuple.append([item, "Phenotype"])

fea_map = pd.DataFrame(tmp_feat_tuple, columns=["feature_name", "feature_type"])
fea_map["feature_name"] = fea_map["feature_name"].astype(str)
fea_map["feature_name_std"] = fea_map["feature_name"].apply(
    lambda x: x if x.startswith("fea_") else f"fea_{x}"
)
feature_type_map = dict(zip(fea_map["feature_name_std"], fea_map["feature_type"]))

# =========================================================
# 画图风格
# =========================================================
sns.set_theme(style="white", font_scale=1.1)
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42


# =========================================================
# 函数1：规范 year，去掉 .0
# =========================================================
def clean_year_value(y):
    return str(int(float(y)))


# =========================================================
# 函数2：计算 SHAP 排名
# =========================================================
def get_shap_rank_df(model, data_df, feature_cols, cat_features, keep_only_fea=True):
    pool = Pool(
        data=data_df[feature_cols],
        cat_features=cat_features
    )

    shap_values = model.get_feature_importance(pool, type="ShapValues")
    shap_values = np.array(shap_values)

    # CatBoost SHAP 最后一列是 expected value
    shap_values = shap_values[:, :-1]

    mean_abs_shap = np.abs(shap_values).mean(axis=0)

    shap_df = pd.DataFrame({
        "feature": feature_cols,
        "mean_abs_shap": mean_abs_shap
    })

    if keep_only_fea:
        shap_df = shap_df[shap_df["feature"].str.startswith("fea_")].copy()

    shap_df = shap_df.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
    shap_df["rank"] = np.arange(1, len(shap_df) + 1)

    return shap_df


# =========================================================
# 函数3：绘制 heatmap
# =========================================================
def plot_rank_heatmap(rank_df, out_png, heatmap_cmap="YlGnBu_r", dpi=600):
    n_rows, n_cols = rank_df.shape
    fig_width = max(10, n_cols * 0.8 + 3)
    fig_height = max(6, n_rows * 0.38 + 1.5)

    fig, ax = plt.subplots(figsize=(fig_width, fig_height))

    sns.heatmap(
        rank_df,
        ax=ax,
        cmap=heatmap_cmap,
        annot=True,
        fmt=".0f",
        linewidths=0.6,
        linecolor="white",
        cbar=False,
        square=False
    )

    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    fig.savefig(out_png, dpi=dpi, bbox_inches="tight")
    plt.close(fig)


# =========================================================
# 主程序
# =========================================================
all_rank_tables = []
all_whole_top10_tables = []

for target_col in traits:

    print(f"\n========== Running trait: {target_col} ==========")

    train_file = os.path.join(IN_PATH, f"train-{target_col}.txt")
    if not os.path.exists(train_file):
        print(f"Skip {target_col}: file not found -> {train_file}")
        continue

    train_df = pd.read_table(train_file)

    train_df["location"] = train_df["location"].astype(str)
    train_df["year"] = train_df["year"].apply(clean_year_value)

    fea_cols = [c for c in train_df.columns if c.startswith("fea_")]
    cat_features = ["location", "year"]
    model_feature_cols = fea_cols + cat_features

    # year × location 组合
    pairs = (
        train_df[["year", "location"]]
        .drop_duplicates()
        .sort_values(["year", "location"])
    )
    pairs = list(pairs.itertuples(index=False, name=None))

    # =====================================================
    # TrainingWhole
    # =====================================================
    whole_pool = Pool(
        data=train_df[model_feature_cols],
        label=train_df[target_col],
        cat_features=cat_features
    )

    whole_model = CatBoostRegressor(
        iterations=1000,
        thread_count=48,
        verbose=0,
        random_seed=42
    )
    whole_model.fit(whole_pool)

    # Whole SHAP 排名
    whole_shap_rank = get_shap_rank_df(
        model=whole_model,
        data_df=train_df,
        feature_cols=model_feature_cols,
        cat_features=cat_features,
        keep_only_fea=True
    )

    # 取前10
    whole_top10_df = whole_shap_rank.head(10).copy()
    top10_features = whole_top10_df["feature"].tolist()

    whole_top10_df["Trait"] = target_col
    whole_top10_df["feature_display"] = whole_top10_df["feature"].str.replace("^fea_", "", regex=True)
    whole_top10_df["feature_type"] = whole_top10_df["feature"].map(feature_type_map).fillna("Unknown")
    all_whole_top10_tables.append(whole_top10_df.copy())

    # =====================================================
    # TrainingTwice 第一阶段：先用全数据训练
    # =====================================================
    first_model = CatBoostRegressor(
        iterations=1000,
        thread_count=48,
        verbose=0,
        random_seed=42
    )
    first_model.fit(whole_pool)

    # =====================================================
    # 对每个环境做 TrainingTwice，再提取 top10 特征排名
    # =====================================================
    rank_matrix = []
    row_names = []
    detail_records = []

    for year, location in pairs:

        year_clean = clean_year_value(year)

        train_data = train_df[
            (train_df["year"] == year) &
            (train_df["location"] == location)
        ].copy()

        # 样本太少时跳过
        if len(train_data) < 5:
            print(f"Skip {year_clean}_{location}: too few samples ({len(train_data)})")
            continue

        train_pool = Pool(
            data=train_data[model_feature_cols],
            label=train_data[target_col],
            cat_features=cat_features
        )

        sec_model = CatBoostRegressor(
            iterations=1000,
            thread_count=48,
            verbose=0,
            random_seed=42
        )

        sec_model.fit(train_pool, init_model=first_model)

        # 用局部环境数据计算 SHAP 排名
        sec_shap_rank = get_shap_rank_df(
            model=sec_model,
            data_df=train_data,
            feature_cols=model_feature_cols,
            cat_features=cat_features,
            keep_only_fea=True
        )

        sec_rank_map = dict(zip(sec_shap_rank["feature"], sec_shap_rank["rank"]))

        # 提取 whole top10 在当前模型中的排名
        current_ranks = [sec_rank_map.get(f, np.nan) for f in top10_features]
        rank_matrix.append(current_ranks)
        row_names.append(f"CB_Dynamic({year_clean}_{location})")

        # 保存详细表
        for feat in top10_features:
            detail_records.append({
                "Trait": target_col,
                "year": int(year_clean),
                "location": location,
                "model_name": f"CB_Dynamic({year_clean}_{location})",
                "feature": feat,
                "feature_display": feat.replace("fea_", ""),
                "feature_type": feature_type_map.get(feat, "Unknown"),
                "Whole_rank": int(whole_top10_df.loc[whole_top10_df["feature"] == feat, "rank"].iloc[0]),
                "Twice_rank": sec_rank_map.get(feat, np.nan)
            })

    if len(rank_matrix) == 0:
        print(f"No valid TrainingTwice models for {target_col}")
        continue

    # =====================================================
    # 组装 rank matrix
    # 列顺序严格按 TrainingWhole top10 的 rank 顺序
    # 并在最后增加一行 CB_FitAll = 1~10
    # =====================================================
    heatmap_df = pd.DataFrame(
        rank_matrix,
        index=row_names,
        columns=whole_top10_df["feature_display"].tolist()
    )

    whole_rank_row = pd.DataFrame(
        [list(range(1, len(whole_top10_df) + 1))],
        index=["CB_FitAll"],
        columns=heatmap_df.columns
    )

    heatmap_df = pd.concat([heatmap_df, whole_rank_row], axis=0)

    # =====================================================
    # 输出矩阵
    # =====================================================
    heatmap_tsv = os.path.join(OUTDIR, f"{target_col}_TrainingTwice_top10_rank_matrix.tsv")
    heatmap_df.to_csv(heatmap_tsv, sep="\t")

    detail_df = pd.DataFrame(detail_records)
    detail_tsv = os.path.join(OUTDIR, f"{target_col}_TrainingTwice_top10_rank_detail.tsv")
    detail_df.to_csv(detail_tsv, sep="\t", index=False)

    all_rank_tables.append(detail_df)

    # =====================================================
    # 作图
    # =====================================================
    out_png = os.path.join(OUTDIR, f"{target_col}_TrainingTwice_top10_rank_heatmap.png")
    out_pdf = os.path.join(OUTDIR, f"{target_col}_TrainingTwice_top10_rank_heatmap.pdf")

    plot_rank_heatmap(
        rank_df=heatmap_df,
        out_png=out_png,
        heatmap_cmap="YlGnBu_r",
        dpi=600
    )

    plot_rank_heatmap(
        rank_df=heatmap_df,
        out_png=out_pdf,
        heatmap_cmap="YlGnBu_r",
        dpi=600
    )

    print(f"Saved heatmap: {out_png}")
    print(f"Saved heatmap: {out_pdf}")
    print(f"Saved matrix : {heatmap_tsv}")
    print(f"Saved detail : {detail_tsv}")
    
# =========================================================
# 汇总输出
# =========================================================
if len(all_rank_tables) > 0:
    all_rank_df = pd.concat(all_rank_tables, axis=0, ignore_index=True)
    all_rank_df.to_csv(
        os.path.join(OUTDIR, "ALL_traits_TrainingTwice_top10_rank_detail.tsv"),
        sep="\t",
        index=False
    )

if len(all_whole_top10_tables) > 0:
    all_whole_top10_df = pd.concat(all_whole_top10_tables, axis=0, ignore_index=True)
    all_whole_top10_df.to_csv(
        os.path.join(OUTDIR, "ALL_traits_TrainingWhole_top10_features.tsv"),
        sep="\t",
        index=False
    )

print("\nAll done.")


========== Running trait: Yield_per_plant ==========
Saved heatmap: /data2/zhoujb/project/cropVarCefComb/processData/9-0-runOnline/trainTestData/shap_rank_heatmap_sidecolor/Yield_per_plant_TrainingTwice_top10_rank_heatmap.png
Saved heatmap: /data2/zhoujb/project/cropVarCefComb/processData/9-0-runOnline/trainTestData/shap_rank_heatmap_sidecolor/Yield_per_plant_TrainingTwice_top10_rank_heatmap.pdf
Saved matrix : /data2/zhoujb/project/cropVarCefComb/processData/9-0-runOnline/trainTestData/shap_rank_heatmap_sidecolor/Yield_per_plant_TrainingTwice_top10_rank_matrix.tsv
Saved detail : /data2/zhoujb/project/cropVarCefComb/processData/9-0-runOnline/trainTestData/shap_rank_heatmap_sidecolor/Yield_per_plant_TrainingTwice_top10_rank_detail.tsv

========== Running trait: 1000-grain_weight ==========
Saved heatmap: /data2/zhoujb/project/cropVarCefComb/processData/9-0-runOnline/trainTestData/shap_rank_heatmap_sidecolor/1000-grain_weight_TrainingTwice_top10_rank_heatmap.png
Saved heatmap: /data2/zho